# Customer Segmentation (RFM)

Segment customers based on:

• Recency (how recently a customer purchased)
• Frequency (number of purchases)
• Monetary (total revenue)

This helps identify high-value and at-risk customers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/clean_retail.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [ ]:
orders = (
    df.groupby("Invoice", as_index=False)
      .agg(
          OrderValue=("Revenue","sum"),
          OrderDate=("InvoiceDate","min"),
          CustomerID=("Customer ID","first")
      )
)

In [ ]:
snapshot_date = orders["OrderDate"].max() + pd.Timedelta(days=1)

In [ ]:
rfm = (
    orders.groupby("CustomerID")
    .agg({
        "OrderDate": lambda x: (snapshot_date - x.max()).days,
        "Invoice": "nunique",
        "OrderValue": "sum"
    })
)

rfm.columns = ["Recency","Frequency","Monetary"]

rfm.head()

In [ ]:
rfm["R_score"] = pd.qcut(rfm["Recency"],4,labels=[4,3,2,1])
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"),4,labels=[1,2,3,4])
rfm["M_score"] = pd.qcut(rfm["Monetary"],4,labels=[1,2,3,4])

In [ ]:
rfm["RFM_score"] = (
    rfm["R_score"].astype(int) +
    rfm["F_score"].astype(int) +
    rfm["M_score"].astype(int)
)

In [ ]:
def segment_customer(score):

    if score >= 10:
        return "VIP"

    if score >= 7:
        return "Loyal"

    if score >= 5:
        return "Potential"

    return "At Risk"


rfm["Segment"] = rfm["RFM_score"].apply(segment_customer)

rfm.head()

In [ ]:
segment_counts = rfm["Segment"].value_counts()

segment_counts

In [ ]:
plt.figure(figsize=(8,4))

segment_counts.plot(kind="bar")

plt.title("Customer Segmentation (RFM)")
plt.xlabel("Segment")
plt.ylabel("Number of Customers")

plt.show()